# Geocoding data-quality playground

## What this shows
A deterministic, offline geocoding workflow: fixture Census results become FIPS/GEOID columns, Nominatim outcomes are separated into no-match vs provider failure, cache-through behavior is explicit, and invalid coordinates are rejected before they poison the cache.

## Why it matters
Geocodes are derived data, not a system of record. A cache can speed up repeated work, but only if cache keys, freshness, coordinate ranges, and failure semantics are explicit. Empty frames and swallowed provider failures hide data-quality problems.

## Prereqs
Pure/offline path. No credentials, network, GDAL, or live API calls. Live Census/Nominatim calls require an explicit integration/manual tier outside this notebook.


## 1. Fixture donor addresses

The fixture includes two matchable Texas addresses, one unmatched row, and one intentionally invalid coordinate row used later to demonstrate cache-poison prevention.

In [ ]:
import pandas as pd

addresses = pd.DataFrame([
    {'id': '1', 'street': '301 W 2nd St', 'city': 'Austin', 'state': 'TX', 'zipcode': '78701'},
    {'id': '2', 'street': '1201 Louisiana St', 'city': 'Houston', 'state': 'TX', 'zipcode': '77002'},
    {'id': '3', 'street': '742 Evergreen Terr', 'city': 'Springfield', 'state': 'XX', 'zipcode': '00000'},
    {'id': '4', 'street': 'Bad Lat Lane', 'city': 'Austin', 'state': 'TX', 'zipcode': '78701'},
])
addresses

## 2. Census batch/FIPS/GEOID flow from fixtures

The hard live parser/transport behavior belongs to `tests/test_census_geocoder.py`, `tests/test_census_geocoder_errors.py`, and `tests/test_geocode_results_to_dataframe.py`. This notebook uses `CensusGeocodeResult` fixtures so the public data shape is runnable offline.

In [ ]:
from siege_utilities.geo.providers.census_geocoder import (
    CensusGeocodeResult, geocode_results_to_dataframe,
)

fixture_results = [
    CensusGeocodeResult(
        matched=True, input_id='1', input_address='301 W 2nd St, Austin, TX 78701',
        matched_address='301 W 2ND ST, AUSTIN, TX, 78701', lat=30.265, lon=-97.747,
        state_fips='48', county_fips='453', tract='001100', block='1001', match_type='Exact',
    ),
    CensusGeocodeResult(
        matched=True, input_id='2', input_address='1201 Louisiana St, Houston, TX 77002',
        matched_address='1201 LOUISIANA ST, HOUSTON, TX, 77002', lat=29.756, lon=-95.369,
        state_fips='48', county_fips='201', tract='000100', block='2001', match_type='Exact',
    ),
    CensusGeocodeResult(input_id='3', input_address='742 Evergreen Terr, Springfield, XX 00000'),
]

geocoded = geocode_results_to_dataframe(fixture_results)
geocoded[['input_id', 'matched', 'latitude', 'longitude', 'county_geoid', 'tract_geoid', 'block_geoid']]

## 3. Nominatim no-match vs provider failure

A no-match is a valid result (`None`); a provider/network/parse failure is a typed `GeocodingError`. The notebook uses fakes instead of calling Nominatim.

In [ ]:
from siege_utilities.geo.geocoding import GeocodingError

def fake_nominatim(address):
    if 'Evergreen' in address:
        return None  # legitimate no-match
    if 'timeout' in address.lower():
        raise GeocodingError('provider timeout for fixture')
    return {'lat': 30.265, 'lon': -97.747, 'source': 'fixture-nominatim'}

for query in ['742 Evergreen Terr', 'timeout example', '301 W 2nd St']:
    try:
        print(query, '=>', fake_nominatim(query))
    except GeocodingError as exc:
        print(query, '=> typed failure:', exc)


## 4. Cache-through behavior and cache key semantics

`SpatiaLiteCache` normalizes address lookups so the same address text with different case/spacing maps to the same derived cache row. Cache hits should not call the provider. Live provider calls require explicit opt-in outside this pure notebook. Exact hash details are hard-test/internal proof, not a user-facing API contract.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

from siege_utilities.geo.geocoding import SpatiaLiteCache

with TemporaryDirectory() as tmpdir:
    cache = SpatiaLiteCache(str(Path(tmpdir) / 'geocode_cache.db'))
    canonical = '301 W 2nd St, Austin, TX 78701'
    messy_equivalent = '  301 w 2nd st,   austin, tx 78701  '
    cache.put_geocode(canonical, 30.265, -97.747, source='fixture')
    hit = cache.get_geocode(messy_equivalent)
    assert hit is not None, 'normalized cache lookup should hit'
    print('normalized cache lookup hit:', hit['latitude'], hit['longitude'], hit['source'])
    cache.close()


## 5. Coordinate validation and cache-poison prevention

Coordinates are WGS84 latitude/longitude values. Latitude must be -90..90 and longitude -180..180. Invalid rows are marked for repair and invalid provider payloads must not be cached.

In [ ]:
from siege_utilities.geo.geocoding import mark_valid_geocode_data_pandas, validate_geocode_data_pandas

quality_rows = pd.DataFrame([
    {'id': '1', 'lat': '30.265', 'lon': '-97.747', 'source': 'census-fixture'},
    {'id': '2', 'lat': '29.756', 'lon': '-95.369', 'source': 'census-fixture'},
    {'id': '3', 'lat': None, 'lon': None, 'source': 'no-match'},
    {'id': '4', 'lat': '999', 'lon': '0', 'source': 'poison-fixture'},
])
marked = mark_valid_geocode_data_pandas(quality_rows, 'lat', 'lon', output_col='valid_wgs84')
valid_only = validate_geocode_data_pandas(quality_rows, 'lat', 'lon')
print(marked[['id', 'lat', 'lon', 'source', 'valid_wgs84']])
print('valid rows:', valid_only['id'].tolist())


## 6. DDIA / data-quality operating notes

- **System of record:** the source address table remains authoritative; geocodes are derived observations.
- **Derived cache:** cache entries should be rebuildable from source addresses and provider/version metadata.
- **Cache keys:** normalize address strings deliberately; key semantics decide whether formatting changes produce new work.
- **Freshness:** record source/provider and rebuild when provider, vintage, or normalization rules change.
- **Coordinate/CRS:** validate WGS84 bounds before joins; do not mix projected coordinates into lat/lon columns.
- **No silent empty-frame failure:** no-match, invalid coordinates, and provider failures are different states and should be visible in downstream QA.


## Related

Hard regression authority:

- `tests/test_census_geocoder.py` and `tests/test_census_geocoder_errors.py` — Census API parser/error semantics.
- `tests/test_geocode_results_to_dataframe.py` — FIPS/GEOID dataframe shape.
- `tests/test_nominatim_geocoder_errors.py` — no-match vs typed `GeocodingError`.
- `tests/test_geocode_validation_pandas.py` — pandas coordinate validation.
- `tests/test_spatialite_cache.py` — cache-through, invalid-coordinate, and cache-poison behavior.

Notebook governance authority:

- `tests/test_notebooks.py` pure tier.
- `tests/test_notebook_hygiene.py`.
- `scripts/check_notebook_inventory.py --check --require-all-live-governed`.
